In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t

import ft_digital_trade.src.utils.read_data as read_utils
import ft_digital_trade.src.utils.clean_utils as clean_utils
import ft_digital_trade.src.utils.calculation_utils as calc_utils
import ft_digital_trade.src.utils.plot_utils as plot_utils

client = bigquery.Client()

In [ ]:
# Builds off of the Analysis for section 4 work
# The "online_mcg_totals_quarterly.csv" is the following breakdown of all mcg's (including 'All'):
# - adjusted online spend
# - nominal change Q on Q
# - % change Q on Q
# - contribution to all change Q on Q
df = pd.read_csv('online_mcg_totals_quarterly.csv')
df = df[df['mcg'] != 'All']
df = df.drop('Unnamed:0])

In [ ]:
# Convert time_period_value to datetime-like format for easier filtering
df['year'] = df['time_period_value'].str[:4]
df['quarter'] = df['time_period_value'].str[4:]
df['year'] = df['year'].astype(int)
df

In [ ]:
# Filter for relevant years
df_2019 = df[df['year'] == 2019]
df_2024Q1 = df[df['time_period_value'] == '2024Q1']
df_2025Q1 = df[df['time_period_value'] == '2025Q1']

# Merge 2025Q1 with 2019 and 2024Q1 for comparison
df_2019_base = df_2019.groupby('mcg')['adjusted_online_spend'].sum().reset_index().rename(columns={'adjusted_online_spend': 'spend_2019'})
df_2024Q1_base = df_2024Q1[['mcg', 'adjusted_online_spend']].rename(columns={'adjusted_online_spend': 'spend_2024Q1'})
df_2025Q1_base = df_2025Q1[['mcg', 'adjusted_online_spend']].rename(columns={'adjusted_online_spend': 'spend_2025Q1'})

# Merge all together
merged = df_2025Q1_base.merge(df_2019_base, on='mcg', how='left').merge(df_2024Q1_base, on='mcg', how='left')

# Calculate changes
merged['change_since_2019'] = merged['spend_2025Q1'] - merged['spend_2019']
merged['change_since_2024Q1'] = merged['spend_2025Q1'] - merged['spend_2024Q1']

# Largest MCG in 2025Q1
largest_mcg_2025Q1 = merged.loc[merged['spend_2025Q1'].idxmax(), ['mcg', 'change_since_2019']]

# Biggest increase since 2019
biggest_rise_2019 = merged.loc[merged['change_since_2019'].idxmax(), ['mcg', 'change_since_2019']]

# Biggest increase since 2024Q1
biggest_rise_2024Q1 = merged.loc[merged['change_since_2024Q1'].idxmax(), ['mcg', 'change_since_2024Q1']]

# Biggest fall since 2019
biggest_fall_2019 = merged.loc[merged['change_since_2019'].idxmin(), ['mcg', 'change_since_2019']]

# Biggest fall since 2024Q1
biggest_fall_2024Q1 = merged.loc[merged['change_since_2024Q1'].idxmin(), ['mcg', 'change_since_2024Q1']]

# Display results
print("Largest MCG in 2025Q1 and change since 2019:\n", largest_mcg_2025Q1)
print("\nBiggest rise since 2019:\n", biggest_rise_2019)
print("\nBiggest rise since 2024Q1:\n", biggest_rise_2024Q1)
print("\nBiggest fall since 2019:\n", biggest_fall_2019)
print("\nBiggest fall since 2024Q1:\n", biggest_fall_2024Q1)
